# 02 - NRI Data Pipeline

Takes the long-format NRI data produced by `01_nri_explore.ipynb` and prepares it for joining to the analysis dataset.

**Input:** `NRI_Long.csv` — one row per county per NRI vintage (2020, 2021, 2023, 2025)

**Output:** `nri_panel.parquet` — one row per county per storm year (2020–2025), with NRI scores vintage-matched to each year

**Vintage matching rule (biennial release schedule):**
- 2020 → NRI 2020
- 2021 → NRI 2021
- 2022 → NRI 2021 (forward-fill, no 2022 release)
- 2023 → NRI 2023
- 2024 → NRI 2023 (forward-fill, no 2024 release)
- 2025 → NRI 2025

**Time-adjusted NRI**
- November 2020 (Using for Jan-Nov, no other reference)
- November 2021
- March 2023
- December 2025

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load

In [2]:
nri_raw = pd.read_csv('../data/processed/NRI_Long.csv', index_col=0)

print(f'Shape: {nri_raw.shape}')
print(f'Vintages: {sorted(nri_raw["year"].unique())}')
print(f'Unique counties: {nri_raw["nri_id"].nunique()}')
nri_raw.head()

Shape: (12576, 11)
Vintages: [np.int64(2020), np.int64(2021), np.int64(2023), np.int64(2025)]
Unique counties: 3144


,nri_id,statefips,countyfips,stateabbrv,county,risk_value,eal_valt,sovi_score,resl_score,crf_value,year
0,C01001,1,1,AL,Autauga,2.602012e+06,3.311627e+06,25.857312,55.529800,0.785720,2020
1,C01001,1,1,AL,Autauga,2.087423e+06,2.656700e+06,25.857312,55.529800,0.785720,2021
2,C01001,1,1,AL,Autauga,6.156054e+06,5.514047e+06,51.299999,51.810001,1.116431,2023
3,C01001,1,1,AL,Autauga,2.051090e+07,1.975657e+07,38.040712,55.120865,1.038181,2025
4,C01003,1,3,AL,Baldwin,1.834143e+07,1.816858e+07,34.292471,54.759202,1.009514,2020


## Clean and derive FIPS

In [3]:
nri = nri_raw.copy()

# Derive 5-digit FIPS from nri_id (format: 'C' + FIPS)
nri['stcofips'] = nri['nri_id'].str[1:]

# Parse into state and county FIPS components for joining
nri['state_fips'] = nri['stcofips'].str[:2]
nri['county_fips'] = nri['stcofips'].str[2:]

# Rename year to nri_vintage to distinguish from storm year
nri = nri.rename(columns={'year': 'nri_vintage'})

# Drop redundant index column
nri = nri.drop(columns=['nri_id'])

print(nri.dtypes)
nri.head()

statefips        int64
countyfips       int64
stateabbrv      object
county          object
risk_value     float64
eal_valt       float64
sovi_score     float64
resl_score     float64
crf_value      float64
nri_vintage      int64
stcofips        object
state_fips      object
county_fips     object
dtype: object


,statefips,countyfips,stateabbrv,county,risk_value,eal_valt,sovi_score,resl_score,crf_value,nri_vintage,stcofips,state_fips,county_fips
0,1,1,AL,Autauga,2.602012e+06,3.311627e+06,25.857312,55.529800,0.785720,2020,01001,01,001
1,1,1,AL,Autauga,2.087423e+06,2.656700e+06,25.857312,55.529800,0.785720,2021,01001,01,001
2,1,1,AL,Autauga,6.156054e+06,5.514047e+06,51.299999,51.810001,1.116431,2023,01001,01,001
3,1,1,AL,Autauga,2.051090e+07,1.975657e+07,38.040712,55.120865,1.038181,2025,01001,01,001
4,1,3,AL,Baldwin,1.834143e+07,1.816858e+07,34.292471,54.759202,1.009514,2020,01003,01,003


## Validate — no nulls, FIPS formatting correct

In [4]:
assert nri.isnull().sum().sum() == 0, 'Unexpected nulls in NRI data'
assert nri['stcofips'].str.len().eq(5).all(), 'FIPS not all 5 digits'
assert nri['state_fips'].str.len().eq(2).all(), 'State FIPS not all 2 digits'
assert nri['county_fips'].str.len().eq(3).all(), 'County FIPS not all 3 digits'
assert set(nri['nri_vintage'].unique()) == {2020, 2021, 2023, 2025}, 'Unexpected vintages'

print('All assertions passed')

All assertions passed


## Expand to storm months and years via vintage matching smoothed across releases
Set NRI release months as anchors, cross join counties x months, interpolate scores between anchors

In [20]:
SCORE_COLS = ['risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']

# Release anchor: (year, month, nri_vintage)
RELEASE_ANCHORS = [
    (2020, 11, 2020),
    (2021, 11, 2021),
    (2023,  3, 2023),
    (2025, 12, 2025),
]

# --- Build anchor rows -------------------------------------------------------
# One row per county per anchor month, with actual NRI values at that month
anchor_rows = []
for yr, mo, vintage in RELEASE_ANCHORS:
    anchor_date = pd.Timestamp(year=yr, month=mo, day=1)
    rows = (
        nri[nri['nri_vintage'] == vintage][['stcofips'] + SCORE_COLS]
        .copy()
        .assign(month=anchor_date, nri_vintage=vintage)
    )
    anchor_rows.append(rows)

anchors = pd.concat(anchor_rows, ignore_index=True)

# --- Cross-join counties x months -------------------------------------------
county_ids = nri[['stcofips', 'state_fips', 'county_fips', 'stateabbrv', 'county']].drop_duplicates('stcofips')
all_months = pd.date_range('2020-01', '2025-12', freq='MS')

county_months = county_ids.merge(
    pd.DataFrame({'month': all_months}), how='cross'
)

# --- Merge anchors, then interpolate ----------------------------------------
nri_panel = (
    county_months
    .merge(anchors, on=['stcofips', 'month'], how='left')
    .sort_values(['stcofips', 'month'])
    .reset_index(drop=True)
)

# interpolate fills between anchors; bfill covers Jan–Oct 2020 (before first anchor)
nri_panel[SCORE_COLS] = (
    nri_panel
    .groupby('stcofips', sort=False)[SCORE_COLS]
    .transform(lambda s: s.interpolate(method='index').bfill())
)

# nri_vintage is only set at anchor months — forward-fill to label each month
nri_panel['nri_vintage'] = (
    nri_panel.groupby('stcofips')['nri_vintage']
    .transform(lambda s: s.ffill().bfill())
    .astype(int)
)
print(f'Panel shape: {nri_panel.shape}')
print(f'Expected rows: {county_ids.shape[0]} counties x 6 years x 12 months = {county_ids.shape[0] * 6 * 12}')
nri_panel.head(12)

Panel shape: (226368, 12)
Expected rows: 3144 counties x 6 years x 12 months = 226368


,stcofips,state_fips,county_fips,stateabbrv,county,month,risk_value,eal_valt,sovi_score,resl_score,crf_value,nri_vintage
0,01001,01,001,AL,Autauga,2020-01-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
1,01001,01,001,AL,Autauga,2020-02-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
2,01001,01,001,AL,Autauga,2020-03-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
3,01001,01,001,AL,Autauga,2020-04-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
4,01001,01,001,AL,Autauga,2020-05-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
5,01001,01,001,AL,Autauga,2020-06-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
6,01001,01,001,AL,Autauga,2020-07-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
7,01001,01,001,AL,Autauga,2020-08-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
8,01001,01,001,AL,Autauga,2020-09-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020
9,01001,01,001,AL,Autauga,2020-10-01,2.602012e+06,3.311627e+06,25.857312,55.5298,0.78572,2020


In [21]:
assert nri_panel.isnull().sum().sum() == 0, 'Nulls remain after interpolation'
assert nri_panel.duplicated(['stcofips', 'month']).sum() == 0, 'Duplicate county-month rows'
assert nri_panel['month'].min() == pd.Timestamp('2020-01-01')
assert nri_panel['month'].max() == pd.Timestamp('2025-12-01')

# Spot-check: anchor months should match raw NRI values exactly
spot = nri_panel[nri_panel['month'] == pd.Timestamp('2023-03-01')]
raw_2023 = nri[nri['nri_vintage'] == 2023].set_index('stcofips')
for col in SCORE_COLS:
    assert np.allclose(spot.set_index('stcofips')[col], raw_2023[col]), f'{col} mismatch at 2023-03'

print('All assertions passed')

All assertions passed


## Summary statistics by vintage

In [24]:
nri_panel.groupby('nri_vintage')[['resl_score', 'risk_value', 'eal_valt', 'sovi_score']].describe().round(2)

resl_score                                                    \
                 count   mean    std    min    25%    50%    75%     max   
nri_vintage                                                                
2020           69168.0  54.59   2.94  41.19  52.65  54.66  56.75   64.67   
2021           50304.0  52.44  16.74   2.73  42.61  53.43  62.57   97.76   
2023          103752.0  50.01  28.70   0.00  24.98  49.93  74.82  100.00   
2025            3144.0  50.02  28.87   0.03  25.02  50.02  75.01  100.00   

            risk_value               ...     eal_valt                \
                 count         mean  ...          75%           max   
nri_vintage                          ...                              
2020           69168.0  11814340.47  ...   8365173.49  1.431320e+09   
2021           50304.0  19117245.01  ...   9846272.24  3.761047e+09   
2023          103752.0  39307396.92  ...  22626112.34  7.490160e+09   
2025            3144.0  51500133.49  ...  32507060.81  7.601846e+09   

            sovi_score                                                   
                 count   mean    std    min    25%    50%    75%    max  
nri_vintage                                                              
2020           69168.0  38.35  11.14   0.01  31.88  38.33  44.46  100.0  
2021           50304.0  43.82  18.87   0.01  30.51  42.04  55.67  100.0  
2023          103752.0  50.31  26.58   0.00  27.43  50.12  72.73  100.0  
2025            3144.0  50.62  28.00  10.62  25.05  50.03  75.03  100.0  

[4 rows x 32 columns]

## Export

In [26]:
output_cols = [
    'stcofips', 'state_fips', 'county_fips', 'stateabbrv', 'county',
    'month', 'nri_vintage',
    'resl_score', 'risk_value', 'eal_valt', 'sovi_score', 'crf_value'
]

nri_panel = nri_panel[output_cols].sort_values(['stcofips', 'month']).reset_index(drop=True)

nri_panel.to_pickle('../data/processed/nri_panel.pkl')

print(f'Exported nri_panel.parquet')
print(f'Shape: {nri_panel.shape}')
print(f'Columns: {nri_panel.columns.tolist()}')

Exported nri_panel.parquet
Shape: (226368, 12)
Columns: ['stcofips', 'state_fips', 'county_fips', 'stateabbrv', 'county', 'month', 'nri_vintage', 'resl_score', 'risk_value', 'eal_valt', 'sovi_score', 'crf_value']
